# PyTorch for LLMs 00: The Deep-Learning Map

> **What you already know:** hidden representations repair linear failures, and backpropagation assigns responsibility to earlier weights.
> **The gap this chapter closes:** the concepts are still disconnected from PyTorch's executable training objects.
> **What you'll have by the end:** one tiny nonlinear regressor trained and used for inference.
> **What this chapter is not:** a DataLoader, checkpoint, or production trainer tutorial.

**Guiding question:** What is the smallest complete route from data to a learned prediction?

\`\`\`text
data (N, 1)
  -> module.forward
prediction (N, 1)
  -> loss(prediction, target)
scalar loss ()
  -> backward
parameter.grad
  -> optimizer.step
updated parameter
\`\`\`

\`\`\`mermaid
flowchart LR
    A["Data"] --> B["Module"]
    B --> C["Forward"]
    C --> D["Loss"]
    D --> E["Backward"]
    E --> F["Optimizer"]
    F --> C
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
\`\`\`


In [ ]:
# -- Imports, reproducibility, and device -------------------------------------
import matplotlib.pyplot as plt
import torch
from torch import nn

torch.manual_seed(7)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__}; device={device}")


## 1. Attempt 1: predict before learning

The running example is a curved sensor response. A network receives one reading and must predict one calibrated value.

**Predict:** Will a freshly initialized network already follow the curve? It has the right shape, not the right parameters.


In [ ]:
# -- Create one tiny learning problem -----------------------------------------
x = torch.linspace(-2.0, 2.0, 160).unsqueeze(1).to(device)
y = (0.55 * x**3 - 0.25 * x + 0.10).to(device)
print(f"x: {tuple(x.shape)} = (example, feature)")
print(f"y: {tuple(y.shape)} = (example, target)")


In [ ]:
# -- Define a small readable module -------------------------------------------
class TinyRegressor(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(1, 12)
        self.output = nn.Linear(12, 1)

    def forward(self, inputs):
        hidden = torch.tanh(self.hidden(inputs))
        return self.output(hidden)

model = TinyRegressor().to(device)
parameter_count = sum(parameter.numel() for parameter in model.parameters())
print(f"Model has {parameter_count} trainable scalar parameters.")


### Code walkthrough: TinyRegressor

- \`nn.Linear(1, 12)\` gives the model twelve candidate hidden features.
- \`tanh\` makes those features nonlinear; without it, two linear layers collapse into one linear map.
- \`nn.Linear(12, 1)\` combines hidden evidence into one target per example.
- \`forward\` states the data path. Calling \`model(x)\` invokes it through \`nn.Module\`.


In [ ]:
# -- Measure the untrained forward contract ----------------------------------
loss_function = nn.MSELoss()
with torch.inference_mode():
    initial_prediction = model(x)
initial_loss = loss_function(initial_prediction, y)
print(f"Forward shape: {tuple(x.shape)} -> {tuple(initial_prediction.shape)}")
print(f"Initial MSE: {initial_loss.item():.6f}")


## 2. One update separates responsibilities

**Predict:** Does \`backward()\` update the weights? No. It fills gradient buffers. The optimizer performs the update.

\`\`\`mermaid
flowchart LR
    A["zero_grad"] --> B["forward"]
    B --> C["loss"]
    C --> D["backward: fill .grad"]
    D --> E["step: change parameters"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
\`\`\`


In [ ]:
# -- Prove what backward and step each change --------------------------------
optimizer = torch.optim.Adam(model.parameters(), lr=0.02)
optimizer.zero_grad()
prediction = model(x)
loss = loss_function(prediction, y)
first_weight_before = model.hidden.weight.detach().clone()
loss.backward()
gradient_norm = model.hidden.weight.grad.norm().item()
unchanged_after_backward = torch.equal(first_weight_before, model.hidden.weight.detach())
optimizer.step()
changed_after_step = not torch.equal(first_weight_before, model.hidden.weight.detach())
print(f"Gradient norm after backward: {gradient_norm:.6f}")
print(f"Weights unchanged after backward: {unchanged_after_backward}")
print(f"Weights changed after optimizer.step: {changed_after_step}")


### That worked, but one local correction is not learning

An epoch repeats the same responsibility cycle over the available data. This first map uses full-batch updates so only the essential loop is visible.


In [ ]:
# -- Repeat the visible loop and record real loss -----------------------------
history = []
for epoch in range(300):
    optimizer.zero_grad()
    prediction = model(x)
    loss = loss_function(prediction, y)
    loss.backward()
    optimizer.step()
    history.append(loss.item())

print(f"Loss: {history[0]:.6f} -> {history[-1]:.6f}")
print(f"Measured reduction factor: {history[0] / history[-1]:.1f}x")


In [ ]:
# -- Plot the measured fit and optimization history ---------------------------
model.eval()
with torch.inference_mode():
    learned_prediction = model(x)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(x.cpu(), y.cpu(), label="target")
axes[0].plot(x.cpu(), learned_prediction.cpu(), label="learned")
axes[0].set(title="Same examples, learned parameters", xlabel="sensor reading")
axes[0].legend()
axes[1].plot(history)
axes[1].set(title="Recorded training loss", xlabel="epoch", ylabel="MSE")
for axis in axes:
    axis.grid(alpha=0.3)
plt.show()
print("The curve and loss history come from the recorded training run.")


## 3. Inference keeps the learned function, not the training graph

Evaluation mode selects inference behavior for modules such as dropout. \`torch.inference_mode()\` also prevents graph construction when no gradients are needed.


In [ ]:
# -- Run inference on readings that were not in the training grid -------------
new_readings = torch.tensor([[-1.25], [0.25], [1.50]], device=device)
model.eval()
with torch.inference_mode():
    new_predictions = model(new_readings)
print("reading -> calibrated prediction")
for reading, predicted in zip(new_readings.cpu(), new_predictions.cpu()):
    print(f"{reading.item():5.2f} -> {predicted.item():6.3f}")
print(f"Inference output requires gradients: {new_predictions.requires_grad}")


### Your turn: change only the epoch budget


In [ ]:
# -- CHANGE THIS: compare a shorter training budget ---------------------------
YOUR_EPOCHS = 40
torch.manual_seed(7)
trial = TinyRegressor().to(device)
trial_optimizer = torch.optim.Adam(trial.parameters(), lr=0.02)
for _ in range(YOUR_EPOCHS):
    trial_optimizer.zero_grad()
    trial_loss = loss_function(trial(x), y)
    trial_loss.backward()
    trial_optimizer.step()
print(f"Loss after {YOUR_EPOCHS} epochs: {trial_loss.item():.6f}")
assert YOUR_EPOCHS > 0


## Scorecard and coverage ledger

| Tier | Covered here |
|---|---|
| Built and measured | data, module, forward, loss, backward, optimizer, epoch, inference |
| Explained or illustrated | why nonlinearity matters; gradient buffers versus parameter updates |
| Named and out of scope | mini-batches, validation, checkpoints, schedulers, mixed precision |

**Common failure modes:** forgetting \`zero_grad\`, assuming \`backward\` changes weights, reporting only a final loss, and building graphs during inference.

**LLM bridge:** an LLM uses the same route. The data shape, module size, loss, and optimizer are larger; the responsibilities do not change.

**Next:** Part 01 opens the tensor and autograd contracts hidden inside this loop.
